In [1]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

# Download the text from the internet
text = requests.get(url).text

# Print the first 500 characters just to see what it looks like
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [2]:
# Let's find out what characters the text uses
chars = sorted(list(set(text)))

print("Total unique characters:", len(chars))
print(chars)


Total unique characters: 65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [3]:
# Convert each character to a number
char_to_num = {ch: i for i, ch in enumerate(chars)}

# Convert numbers back to characters
num_to_char = {i: ch for i, ch in enumerate(chars)}

print("Example mapping:", list(char_to_num.items())[:10])


Example mapping: [('\n', 0), (' ', 1), ('!', 2), ('$', 3), ('&', 4), ("'", 5), (',', 6), ('-', 7), ('.', 8), ('3', 9)]


In [5]:
sequence_length = 40
step = 1

sentences = []    # list of input chunks
next_chars = []   # list of the correct next characters

for i in range(0, len(text) - sequence_length, step):
    # take 40 characters starting at position i
    sentences.append(text[i : i + sequence_length])
    # take the very next character after those 40
    next_chars.append(text[i + sequence_length])



In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Building the model
model = Sequential()

# 1) LSTM Layer
model.add(LSTM(128, input_shape=(sequence_length, len(chars))))

# 2) Output Layer
model.add(Dense(len(chars), activation='softmax'))

# Show model summary (structure)
model.summary()


C:\Users\chala\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        99,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 65)             │         8,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 107,713 (420.75 KB)

 Trainable params: 107,713 (420.75 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.compile(
    loss='categorical_crossentropy', 
    optimizer='adam'
)


In [9]:
import numpy as np

# Create empty arrays for inputs (X) and outputs (y)
X = np.zeros((len(sentences), sequence_length, len(chars)), dtype=np.bool_)
y = np.zeros((len(sentences), len(chars)), dtype=np.bool_)

# Fill X and y with data
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_to_num[char]] = 1
    y[i, char_to_num[next_chars[i]]] = 1


history = model.fit(X, y, batch_size=128, epochs=1)


8714/8714 ━━━━━━━━━━━━━━━━━━━━ 515s 59ms/step - loss: 2.0903


In [ ]:
import numpy as np

def generate_text(model, seed_text, length=300, temperature=1.0):
    """
    Generate text using the trained LSTM model.
    """
    generated = seed_text

    for _ in range(length):
        # Convert current characters to numbers (one-hot)
        x = np.zeros((1, sequence_length, len(chars)))
        for t, char in enumerate(seed_text):
            if char in char_to_num:
                x[0, t, char_to_num[char]] = 1
        
        # Predict next character probabilities
        predictions = model.predict(x, verbose=0)[0]

        # Apply temperature to make text creative
        predictions = np.asarray(predictions).astype('float64')
        predictions = np.log(predictions + 1e-7) / temperature
        probabilities = np.exp/(predictions)  np.sum(np.exp(predictions))

        # Pick next character
        next_char = np.random.choice(list(chars), p=probabilities)

        # Add to result
        generated += next_char

        # Move window forward
        seed_text = seed_text[1:] + next_char

    return generated


In [11]:
seed = text[:40]  # first 40 characters as starting seed
print(generate_text(model, seed, length=500, temperature=0.7))


First Citizen:
Before we proceed any furme for Yat
Whe sures, and thing sace for betes will.

WARINCE:
What mome the.

PRINCES:
Betice my prayes, ore him to the thenes,
And this and have part, godes to the them,
And hemand worce the groves soof apwurt,
What she than I the wandesing to so couse cancetchoud,
I will not theck, my nath nover mestes able
He couss my with me pattering and jomes oofe.

QUEEN ELWARD IV:
Well wescime shang to give to theme are seesterss,
God to her man to my lepsten.

AUTELON:
But thene your buether parcouse o


In [12]:
with open("shakespeare.txt", "w", encoding="utf-8") as f:
    f.write(text)


In [13]:
import pickle

with open("char_to_num.pkl", "wb") as f:
    pickle.dump(char_to_num, f)

with open("num_to_char.pkl", "wb") as f:
    pickle.dump(num_to_char, f)


In [14]:
model.save("lstm_textgen.h5")


In [15]:
with open("shakespeare.txt", "r", encoding="utf-8") as f:
    print("✅ Text saved:", len(f.read()), "characters")
    
with open("char_to_num.pkl", "rb") as f:
    print("✅ char_to_num loaded successfully")

from tensorflow.keras.models import load_model
model = load_model("lstm_textgen.h5")
print("✅ Model loaded successfully")


✅ Text saved: 1115394 characters
✅ char_to_num loaded successfully
✅ Model loaded successfully


In [17]:
import numpy as np

np.save("X.npy", X)
np.save("y.npy", y)


In [19]:
from tensorflow.keras.models import load_model
import pickle
import numpy as np

# Load model
model = load_model("lstm_textgen.h5")

# Load mappings
with open("char_to_num.pkl", "rb") as f:
    char_to_num = pickle.load(f)

with open("num_to_char.pkl", "rb") as f:
    num_to_char = pickle.load(f)

# Load training data (INSTANT, no waiting)
X = np.load("X.npy")
y = np.load("y.npy")

model.compile(loss='categorical_crossentropy', optimizer='adam')